<a href="https://colab.research.google.com/github/2303A52060/AI-Assistant/blob/main/16_1_ASS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Student Information System Schema Design

I will now create the SQL `CREATE TABLE` statements for the `Students`, `Courses`, and `Enrollments` tables, defining primary and foreign keys to establish their relationships.



In [15]:
import sqlite3

# Connect to an in-memory SQLite database for demonstration
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create Students table
cursor.execute('''
CREATE TABLE Students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    date_of_birth DATE,
    email TEXT UNIQUE NOT NULL
);
''')

# Create Courses table
cursor.execute('''
CREATE TABLE Courses (
    course_id INTEGER PRIMARY KEY AUTOINCREMENT,
    course_name TEXT NOT NULL UNIQUE,
    course_code TEXT UNIQUE NOT NULL,
    credits INTEGER
);
''')

# Create Enrollments table (junction table for many-to-many relationship)
cursor.execute('''
CREATE TABLE Enrollments (
    enrollment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    enrollment_date DATE NOT NULL,
    FOREIGN KEY (student_id) REFERENCES Students(student_id),
    FOREIGN KEY (course_id) REFERENCES Courses(course_id),
    UNIQUE (student_id, course_id) -- A student can enroll in a course only once
);
''')

print("Tables created successfully.")

Tables created successfully.


### Generated SQL Queries

Here are the SQL queries as requested:

1.  **Insert a new student record**
2.  **Fetch all courses enrolled by a student**
3.  **Count the number of students in each course**

In [6]:
# 1. Insert a new student record
insert_student_query = """
INSERT INTO Students (first_name, last_name, date_of_birth, email)
VALUES (?, ?, ?, ?);
"""

# Example usage:
cursor.execute(insert_student_query, ('John', 'Doe', '2000-01-15', 'john.doe@example.com'))
cursor.execute(insert_student_query, ('Jane', 'Smith', '1999-05-20', 'jane.smith@example.com'))
cursor.execute(insert_student_query, ('Peter', 'Jones', '2001-11-01', 'peter.jones@example.com'))
conn.commit()

print("Students inserted successfully.")

# Insert some courses
cursor.execute("INSERT INTO Courses (course_name, course_code, credits) VALUES (?, ?, ?);", ('Introduction to Programming', 'CS101', 3))
cursor.execute("INSERT INTO Courses (course_name, course_code, credits) VALUES (?, ?, ?);", ('Database Management', 'CS205', 4))
cursor.execute("INSERT INTO Courses (course_name, course_code, credits) VALUES (?, ?, ?);", ('Calculus I', 'MA101', 3))
conn.commit()
print("Courses inserted successfully.")

# Insert some enrollments
cursor.execute("INSERT INTO Enrollments (student_id, course_id, enrollment_date) VALUES (?, ?, ?);", (1, 1, '2023-09-01'))
cursor.execute("INSERT INTO Enrollments (student_id, course_id, enrollment_date) VALUES (?, ?, ?);", (1, 2, '2023-09-01'))
cursor.execute("INSERT INTO Enrollments (student_id, course_id, enrollment_date) VALUES (?, ?, ?);", (2, 1, '2023-09-01'))
cursor.execute("INSERT INTO Enrollments (student_id, course_id, enrollment_date) VALUES (?, ?, ?);", (3, 2, '2023-09-01'))
cursor.execute("INSERT INTO Enrollments (student_id, course_id, enrollment_date) VALUES (?, ?, ?);", (3, 3, '2023-09-01'))
conn.commit()
print("Enrollments inserted successfully.")


# 2. Fetch all courses enrolled by a specific student (e.g., student_id = 1)
fetch_courses_query = """
SELECT
    C.course_name,
    C.course_code,
    C.credits,
    E.enrollment_date
FROM Enrollments AS E
JOIN Courses AS C ON E.course_id = C.course_id
WHERE E.student_id = ?;
"""

student_id_to_fetch = 1
cursor.execute(fetch_courses_query, (student_id_to_fetch,))
enrolled_courses = cursor.fetchall()

print(f"\nCourses enrolled by student ID {student_id_to_fetch}:")
for course in enrolled_courses:
    print(course)


# 3. Count the number of students in each course
count_students_query = """
SELECT
    C.course_name,
    C.course_code,
    COUNT(E.student_id) AS num_students_enrolled
FROM Courses AS C
LEFT JOIN Enrollments AS E ON C.course_id = E.course_id
GROUP BY C.course_id, C.course_name, C.course_code
ORDER BY C.course_name;
"""

cursor.execute(count_students_query)
students_per_course = cursor.fetchall()

print("\nNumber of students in each course:")
for course_count in students_per_course:
    print(course_count)


Students inserted successfully.
Courses inserted successfully.
Enrollments inserted successfully.

Courses enrolled by student ID 1:
('Introduction to Programming', 'CS101', 3, '2023-09-01')
('Database Management', 'CS205', 4, '2023-09-01')

Number of students in each course:
('Calculus I', 'MA101', 1)
('Database Management', 'CS205', 2)
('Introduction to Programming', 'CS101', 2)


### Task 5: University Examination Database Schema

This schema manages academic performance. It uses a normalized structure to ensure data integrity across students, subjects, and their exam results.

In [13]:
import sqlite3

# Connect to a new in-memory database
exam_conn = sqlite3.connect(':memory:')
exam_cursor = exam_conn.cursor()

# Enable foreign key support in SQLite
exam_cursor.execute("PRAGMA foreign_keys = ON;")

# Create Students table
exam_cursor.execute('''
CREATE TABLE Students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    enrollment_no TEXT UNIQUE NOT NULL
);
''')

# Create Subjects table
exam_cursor.execute('''
CREATE TABLE Subjects (
    subject_id INTEGER PRIMARY KEY AUTOINCREMENT,
    subject_name TEXT NOT NULL UNIQUE,
    subject_code TEXT UNIQUE NOT NULL
);
''')

# Create Exams table (e.g., Midterm, Final)
exam_cursor.execute('''
CREATE TABLE Exams (
    exam_id INTEGER PRIMARY KEY AUTOINCREMENT,
    exam_name TEXT NOT NULL,
    exam_date DATE NOT NULL
);
''')

# Create Results table
exam_cursor.execute('''
CREATE TABLE Results (
    result_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    subject_id INTEGER NOT NULL,
    exam_id INTEGER NOT NULL,
    marks_obtained INTEGER CHECK(marks_obtained >= 0 AND marks_obtained <= 100),
    FOREIGN KEY (student_id) REFERENCES Students(student_id) ON DELETE CASCADE,
    FOREIGN KEY (subject_id) REFERENCES Subjects(subject_id) ON DELETE CASCADE,
    FOREIGN KEY (exam_id) REFERENCES Exams(exam_id) ON DELETE CASCADE
);
''')

print('University Examination schema created successfully.')

University Examination schema created successfully.


In [14]:
# 1. Populate metadata
exam_cursor.execute("INSERT INTO Students (name, enrollment_no) VALUES ('Alice', 'U001'), ('Bob', 'U002')")
exam_cursor.execute("INSERT INTO Subjects (subject_name, subject_code) VALUES ('Data Structures', 'CS201'), ('Algorithms', 'CS202')")
exam_cursor.execute("INSERT INTO Exams (exam_name, exam_date) VALUES ('Final Exam 2024', '2024-05-20')")

# 2. Insert examination results for a student (Alice, ID=1)
results_data = [
    (1, 1, 1, 85), # Alice, DS, Final, 85
    (1, 2, 1, 92), # Alice, Algo, Final, 92
    (2, 1, 1, 78), # Bob, DS, Final, 78
    (2, 2, 1, 88)  # Bob, Algo, Final, 88
]
exam_cursor.executemany("INSERT INTO Results (student_id, subject_id, exam_id, marks_obtained) VALUES (?, ?, ?, ?)", results_data)
exam_conn.commit()
print('Results inserted successfully.\n')

# 3. Retrieve subject-wise marks for a specific student (Alice, ID=1)
print('--- Marks for Student: Alice ---')
exam_cursor.execute('''
SELECT S.subject_name, R.marks_obtained
FROM Results R
JOIN Subjects S ON R.subject_id = S.subject_id
WHERE R.student_id = 1
''')
for row in exam_cursor.fetchall(): print(f"{row[0]}: {row[1]}")

# 4. Calculate average marks for each subject
print('\n--- Average Marks per Subject ---')
exam_cursor.execute('''
SELECT S.subject_name, AVG(R.marks_obtained) as average_score
FROM Subjects S
JOIN Results R ON S.subject_id = R.subject_id
GROUP BY S.subject_id
''')
for row in exam_cursor.fetchall(): print(f"{row[0]}: {row[1]:.2f}")

Results inserted successfully.

--- Marks for Student: Alice ---
Data Structures: 85
Algorithms: 92

--- Average Marks per Subject ---
Data Structures: 81.50
Algorithms: 90.00


### Task 4: E-commerce Platform Database Schema

This schema is designed with **3rd Normal Form (3NF)** principles. I've separated `Orders` and `OrderDetails` to handle multiple products per order efficiently.

**Normalization & Optimization Improvements:**
- **OrderDetails Junction Table**: Prevents data redundancy by allowing a many-to-many relationship between Orders and Products.
- **Indexing**: Added indexes on `order_date` and `user_id` to speed up filtering by time and user.
- **Constraints**: Used `CHECK` constraints to ensure prices and quantities are always positive.

In [11]:
import sqlite3

# Connect to a new in-memory database
shop_conn = sqlite3.connect(':memory:')
shop_cursor = shop_conn.cursor()

# Create Users table
shop_cursor.execute('''
CREATE TABLE Users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL
);
''')

# Create Products table
shop_cursor.execute('''
CREATE TABLE Products (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    price DECIMAL(10, 2) NOT NULL CHECK(price >= 0)
);
''')

# Create Orders table
shop_cursor.execute('''
CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    order_date DATE NOT NULL,
    FOREIGN KEY (user_id) REFERENCES Users(user_id)
);
''')

# Create OrderDetails table
shop_cursor.execute('''
CREATE TABLE OrderDetails (
    detail_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL CHECK(quantity > 0),
    unit_price DECIMAL(10, 2) NOT NULL,
    FOREIGN KEY (order_id) REFERENCES Orders(order_id),
    FOREIGN KEY (product_id) REFERENCES Products(product_id)
);
''')

# Optimization: Indexes
shop_cursor.execute("CREATE INDEX idx_orders_user ON Orders(user_id);")
shop_cursor.execute("CREATE INDEX idx_orders_date ON Orders(order_date);")

print('E-commerce schema and optimization indexes created.')

E-commerce schema and optimization indexes created.


In [12]:
# Sample Data
shop_cursor.execute("INSERT INTO Users (username, email) VALUES ('Alice', 'alice@example.com'), ('Bob', 'bob@example.com')")
shop_cursor.execute("INSERT INTO Products (name, price) VALUES ('Laptop', 1200.00), ('Mouse', 25.00), ('Monitor', 300.00)")

# Orders for March 2024
shop_cursor.execute("INSERT INTO Orders (user_id, order_date) VALUES (1, '2024-03-05'), (1, '2024-03-15'), (2, '2024-03-20')")

# Order Details (Order 1: Laptop+Mouse, Order 2: Mouse, Order 3: Monitor+Mouse)
shop_cursor.execute("INSERT INTO OrderDetails (order_id, product_id, quantity, unit_price) VALUES (1, 1, 1, 1200.00), (1, 2, 1, 25.00), (2, 2, 2, 25.00), (3, 3, 1, 300.00), (3, 2, 1, 25.00)")
shop_conn.commit()

# 1. Retrieve all orders by a user (Alice, ID=1)
print('--- Orders for User: Alice ---')
shop_cursor.execute('''
SELECT O.order_id, O.order_date, P.name, OD.quantity
FROM Orders O
JOIN OrderDetails OD ON O.order_id = OD.order_id
JOIN Products P ON OD.product_id = P.product_id
WHERE O.user_id = 1
''')
for row in shop_cursor.fetchall(): print(row)

# 2. Find the most purchased product (by quantity sold)
print('\n--- Most Purchased Product ---')
shop_cursor.execute('''
SELECT P.name, SUM(OD.quantity) as total_sold
FROM Products P
JOIN OrderDetails OD ON P.product_id = OD.product_id
GROUP BY P.product_id
ORDER BY total_sold DESC
LIMIT 1
''')
print(shop_cursor.fetchone())

# 3. Calculate total revenue in March 2024
print('\n--- Total Revenue (March 2024) ---')
shop_cursor.execute('''
SELECT SUM(quantity * unit_price) as revenue
FROM OrderDetails OD
JOIN Orders O ON OD.order_id = O.order_id
WHERE O.order_date BETWEEN '2024-03-01' AND '2024-03-31'
''')
print(f"${shop_cursor.fetchone()[0]:.2f}")

--- Orders for User: Alice ---
(1, '2024-03-05', 'Laptop', 1)
(1, '2024-03-05', 'Mouse', 1)
(2, '2024-03-15', 'Mouse', 2)

--- Most Purchased Product ---
('Mouse', 4)

--- Total Revenue (March 2024) ---
$1600.00


### Task 3: Library Management System Schema

This schema manages library assets and memberships. I've included an **Indexing Strategy** to optimize common lookups like finding overdue books or searching by member.

In [9]:
import sqlite3
from datetime import datetime, timedelta

# Connect to a new in-memory database
lib_conn = sqlite3.connect(':memory:')
lib_cursor = lib_conn.cursor()

# Create Books table
lib_cursor.execute('''
CREATE TABLE Books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    isbn TEXT UNIQUE NOT NULL,
    total_copies INTEGER DEFAULT 1
);
''')

# Create Members table
lib_cursor.execute('''
CREATE TABLE Members (
    member_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    join_date DATE NOT NULL
);
''')

# Create Loans table
lib_cursor.execute('''
CREATE TABLE Loans (
    loan_id INTEGER PRIMARY KEY AUTOINCREMENT,
    book_id INTEGER NOT NULL,
    member_id INTEGER NOT NULL,
    loan_date DATE NOT NULL,
    return_date DATE, -- NULL if not yet returned
    FOREIGN KEY (book_id) REFERENCES Books(book_id),
    FOREIGN KEY (member_id) REFERENCES Members(member_id)
);
''')

# --- Indexing Strategy ---
# 1. Index on return_date to quickly find currently issued books (where NULL)
lib_cursor.execute("CREATE INDEX idx_loans_return ON Loans(return_date);")
# 2. Index on loan_date to speed up overdue calculations
lib_cursor.execute("CREATE INDEX idx_loans_date ON Loans(loan_date);")
# 3. Index on member_id for fast member history lookups
lib_cursor.execute("CREATE INDEX idx_loans_member ON Loans(member_id);")

print('Library database schema and indexes created successfully.')

Library database schema and indexes created successfully.


### Library Queries

I will populate sample data (including an overdue book) and run the requested queries.

In [10]:
# Sample Data
today = datetime.now().date()
overdue_date = (datetime.now() - timedelta(days=40)).date()

lib_cursor.execute("INSERT INTO Books (title, author, isbn) VALUES ('The Great Gatsby', 'F. Scott Fitzgerald', '9780743273565')")
lib_cursor.execute("INSERT INTO Books (title, author, isbn) VALUES ('1984', 'George Orwell', '9780451524935')")

lib_cursor.execute("INSERT INTO Members (name, email, join_date) VALUES ('Charlie Brown', 'charlie@example.com', '2023-01-01')")
lib_cursor.execute("INSERT INTO Members (name, email, join_date) VALUES ('Lucy van Pelt', 'lucy@example.com', '2023-02-01')")

# Charlie has an overdue book
lib_cursor.execute("INSERT INTO Loans (book_id, member_id, loan_date, return_date) VALUES (1, 1, ?, NULL)", (overdue_date,))
# Lucy has a book issued recently
lib_cursor.execute("INSERT INTO Loans (book_id, member_id, loan_date, return_date) VALUES (2, 2, ?, NULL)", (today,))
lib_conn.commit()

# 1. Retrieve all books currently issued (return_date is NULL)
print('--- Currently Issued Books ---')
lib_cursor.execute('''
SELECT B.title, M.name, L.loan_date
FROM Loans L
JOIN Books B ON L.book_id = B.book_id
JOIN Members M ON L.member_id = M.member_id
WHERE L.return_date IS NULL
''')
for row in lib_cursor.fetchall():
    print(row)

# 2. Find overdue books (loan date > 30 days ago and not returned)
print('\n--- Overdue Books (> 30 Days) ---')
thirty_days_ago = (datetime.now() - timedelta(days=30)).date()
lib_cursor.execute('''
SELECT B.title, M.name, L.loan_date
FROM Loans L
JOIN Books B ON L.book_id = B.book_id
JOIN Members M ON L.member_id = M.member_id
WHERE L.return_date IS NULL AND L.loan_date < ?
''', (thirty_days_ago,))
for row in lib_cursor.fetchall():
    print(row)

# 3. Count number of books loaned by each member
print('\n--- Loan Count per Member ---')
lib_cursor.execute('''
SELECT M.name, COUNT(L.loan_id) as total_loans
FROM Members M
LEFT JOIN Loans L ON M.member_id = L.member_id
GROUP BY M.member_id
''')
for row in lib_cursor.fetchall():
    print(row)

--- Currently Issued Books ---
('The Great Gatsby', 'Charlie Brown', '2026-02-11')
('1984', 'Lucy van Pelt', '2026-03-23')

--- Overdue Books (> 30 Days) ---
('The Great Gatsby', 'Charlie Brown', '2026-02-11')

--- Loan Count per Member ---
('Charlie Brown', 1)
('Lucy van Pelt', 1)


/tmp/ipykernel_746/339326392.py:12: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  lib_cursor.execute("INSERT INTO Loans (book_id, member_id, loan_date, return_date) VALUES (1, 1, ?, NULL)", (overdue_date,))
/tmp/ipykernel_746/339326392.py:14: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  lib_cursor.execute("INSERT INTO Loans (book_id, member_id, loan_date, return_date) VALUES (2, 2, ?, NULL)", (today,))
/tmp/ipykernel_746/339326392.py:32: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  lib_cursor.execute('''


### Task 2: Hospital Management Database Schema

This schema includes constraints for unique IDs, mandatory fields, and valid date relationships.

In [7]:
import sqlite3

# Connect to a new in-memory database
hospital_conn = sqlite3.connect(':memory:')
hospital_cursor = hospital_conn.cursor()

# Create Doctors table
hospital_cursor.execute('''
CREATE TABLE Doctors (
    doctor_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    specialization TEXT,
    license_number TEXT UNIQUE NOT NULL
);
''')

# Create Patients table
hospital_cursor.execute('''
CREATE TABLE Patients (
    patient_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    dob DATE NOT NULL,
    contact_info TEXT
);
''')

# Create Appointments table
hospital_cursor.execute('''
CREATE TABLE Appointments (
    appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    doctor_id INTEGER NOT NULL,
    patient_id INTEGER NOT NULL,
    appointment_date DATETIME NOT NULL,
    diagnosis TEXT,
    FOREIGN KEY (doctor_id) REFERENCES Doctors(doctor_id),
    FOREIGN KEY (patient_id) REFERENCES Patients(patient_id)
);
''')

print('Hospital database schema created successfully.')

Hospital database schema created successfully.


### Hospital Management Queries

I will now populate some sample data and execute the requested queries:
1. List all appointments for a specific doctor.
2. Retrieve patient history by patient ID.
3. Count total patients treated by each doctor.

In [8]:
# Sample Data Insertion
hospital_cursor.execute("INSERT INTO Doctors (name, specialization, license_number) VALUES ('Dr. Smith', 'Cardiology', 'DOC123')")
hospital_cursor.execute("INSERT INTO Doctors (name, specialization, license_number) VALUES ('Dr. Adams', 'Neurology', 'DOC456')")

hospital_cursor.execute("INSERT INTO Patients (name, dob, contact_info) VALUES ('Alice Brown', '1985-04-12', '555-0101')")
hospital_cursor.execute("INSERT INTO Patients (name, dob, contact_info) VALUES ('Bob White', '1992-09-20', '555-0202')")

hospital_cursor.execute("INSERT INTO Appointments (doctor_id, patient_id, appointment_date, diagnosis) VALUES (1, 1, '2023-10-01 10:00:00', 'Checkup')")
hospital_cursor.execute("INSERT INTO Appointments (doctor_id, patient_id, appointment_date, diagnosis) VALUES (1, 2, '2023-10-05 11:30:00', 'Hypertension')")
hospital_cursor.execute("INSERT INTO Appointments (doctor_id, patient_id, appointment_date, diagnosis) VALUES (2, 1, '2023-10-10 09:00:00', 'Migraine')")
hospital_conn.commit()

print('Sample data inserted.\n')

# 1. List all appointments for a specific doctor (Dr. Smith, ID=1)
print('--- Appointments for Dr. Smith ---')
hospital_cursor.execute('''
SELECT A.appointment_date, P.name, A.diagnosis
FROM Appointments A
JOIN Patients P ON A.patient_id = P.patient_id
WHERE A.doctor_id = 1
''')
for row in hospital_cursor.fetchall():
    print(row)

# 2. Retrieve patient history by patient ID (Alice, ID=1)
print('\n--- Patient History for ID 1 ---')
hospital_cursor.execute('''
SELECT A.appointment_date, D.name, A.diagnosis
FROM Appointments A
JOIN Doctors D ON A.doctor_id = D.doctor_id
WHERE A.patient_id = 1
''')
for row in hospital_cursor.fetchall():
    print(row)

# 3. Count total patients treated by each doctor
print('\n--- Patient Count per Doctor ---')
hospital_cursor.execute('''
SELECT D.name, COUNT(DISTINCT A.patient_id) as total_patients
FROM Doctors D
LEFT JOIN Appointments A ON D.doctor_id = A.doctor_id
GROUP BY D.doctor_id
''')
for row in hospital_cursor.fetchall():
    print(row)

Sample data inserted.

--- Appointments for Dr. Smith ---
('2023-10-01 10:00:00', 'Alice Brown', 'Checkup')
('2023-10-05 11:30:00', 'Bob White', 'Hypertension')

--- Patient History for ID 1 ---
('2023-10-01 10:00:00', 'Dr. Smith', 'Checkup')
('2023-10-10 09:00:00', 'Dr. Adams', 'Migraine')

--- Patient Count per Doctor ---
('Dr. Smith', 2)
('Dr. Adams', 1)
